# Pipeline SFT — Logic_Based_Educational_Queries

Code nằm trong `Logic_Based_Educational_Queries_Project/`: `src/services/` (config, Hub, Drive) + `src/models/logic_model/` (SFT + accuracy trên dev). Tài liệu: [`../README.md`](../README.md).

- Flatten 808 câu / 411 record, split **8:1:1** theo `record_id`
- Checkpoint tốt nhất: **`eval_accuracy`** (trắc nghiệm / Yes-No), không dùng `eval_loss`
- Notebook chỉ cài dependency + `%run -i` các stage

## 0. Dependency (chạy khi cần)

In [ ]:
# %pip install -q transformers datasets peft "trl>=0.16.0" accelerate bitsandbytes scikit-learn huggingface_hub gdown python-dotenv
# %pip install -q trl

## 1. Secrets (Kaggle) / token HF

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    
    HF_Token = UserSecretsClient().get_secret("HF_Token")
except Exception:
    HF_Token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or ""

## 2. Bootstrap path cho `services` + `logic_model`

In [ ]:
%run -i project_bootstrap.py

## 3. Cấu hình — chỉnh tại đây

In [ ]:
from services.config import LogicSFTConfig

# Đọc `.env` (sao chép từ `.env.example` → `.env`). Ghi đè tại đây nếu cần.
cfg = LogicSFTConfig.from_env()
print("Config OK", cfg.model_name, cfg.data_source)
print("HF repo (push):", cfg.resolved_hf_repo())

## 4. Tải dữ liệu (chỉ khi `cfg.data_source == "drive"`)
Bỏ qua nếu dùng JSON trong repo (`local`).

In [ ]:
if cfg.data_source == "drive":
    get_ipython().run_line_magic("run", "-i logic_model_stage_drive.py")
else:
    print("DATA_SOURCE=local — bỏ qua tải Drive.")

## 5. Fine-tune (LoRA + chọn model theo accuracy trên dev)

In [ ]:
%run -i logic_model_stage_train.py

## 6. Push Hugging Face (tùy chọn)
Đặt `cfg.push_to_hub = True` và có `HF_Token`.

In [ ]:
%run -i logic_model_stage_push.py